# tsfresh feature generation and t-sne analysis on StressID and ExpData datasets

In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px

from IPython.display import display
from dataclasses import dataclass
from typing import Tuple, TypeAlias

from sklearn.manifold import TSNE
from tsfresh import extract_features, select_features, extract_relevant_features
from tsfresh.feature_extraction.settings import EfficientFCParameters, MinimalFCParameters, IndexBasedFCParameters
from tsfresh.utilities.dataframe_functions import impute

BASE_PATH = "../../.."
DATASET = f"{BASE_PATH}/stressid-dataset"
DATA_ECG = f"{DATASET}/ecg_windowed.csv"
DATA_EDA = f"{DATASET}/eda_windowed.csv"
LABELS_SEPARATOR = ","
LABELS = f"{BASE_PATH}/stressID/labels.csv"
DATA_SEPARATOR = ","
DATA_FS = 500  # Hz
DATA_WINDOW_DURATION = 60  # seconds
TARGET_FS = 51.2
RANDOM_STATE = 21

EDAECG_RELEVANT_FEAT_FILENAME = "../Features/tfresh_edacga_selfeatures.efficient.csv"
EDA_RELEVANT_FEAT_FILENAME = "../Features/tfresh_eda_selfeatures.efficient.csv"
ECG_RELEVANT_FEAT_FILENAME = "../Features/tfresh_ecg_selfeatures.efficient.csv"

BIN_LABELS = ["NoStress", "Stress"]
TER_LABELS = ["Relaxed", "Stress", "RealStress"]
QAD_LABELS = ["Relaxed", "Stress", "RealStress", "Amused"]

LABELS_CONF = {
    "b": {
        "col_name": "binary-stress",
        "enabled": True,
        "stratification": True,
        "classes": BIN_LABELS,
    },
    "t": {
        "col_name": "affect3-class",
        "enabled": False,
        "stratification": True,
        "classes": TER_LABELS,
    },
    "q": {
        "col_name": "affect4-class",
        "enabled": False,
        "stratification": True,
        "classes": QAD_LABELS,
    },
}


@dataclass
class Dataset:
    X: list[pd.Series]
    y: pd.Series
    groups: np.ndarray[int]


CWT: TypeAlias = Tuple[np.ndarray[tuple[int], np.dtype], np.ndarray]

### Build dataset from data files

In [2]:
# Creating labels object

labels_df = pd.read_csv(LABELS, sep=LABELS_SEPARATOR, header=0, index_col=0)
labels: dict[str, pd.Series] = {}
for key, conf in LABELS_CONF.items():
    if conf["enabled"]:
        labels[key] = labels_df[conf["col_name"]]

display(labels["b"])


subject/task
2ea4_Breathing    0
2ea4_Counting1    1
2ea4_Counting2    1
2ea4_Counting3    1
2ea4_Math         1
                 ..
y9z6_Relax        0
y9z6_Speaking     1
y9z6_Stroop       1
y9z6_Video1       1
y9z6_Video2       0
Name: binary-stress, Length: 700, dtype: int64

In [3]:
# Pairing labels and samples for each class type

raw_num_samples = DATA_FS * DATA_WINDOW_DURATION
raw_eda = pd.read_csv(DATA_EDA)
raw_ecg = pd.read_csv(DATA_ECG)

subjects_in_columns: list[int] = [] # subjects (should be 699 after joining eda and ecg)
subject_series: list[int] = []  # "Subject"
task_series: list[str] = []  # "Task"
sample_n_series: list[int] = []  # "Nth-Sample"
eda_series: list[float] = []  # "EDA"
ecg_series: list[float] = []  # "ECG"

blabel_series: list[int] = []  # "b-class"
tlabel_series: list[int] = []  # "t-class"
qlabel_series: list[int] = []  # "q-class"

for class_type, labels_set in labels.items():
    subject_to_group: dict[str, int] = {}
    group_counter = 0
    subjects_list: list[int] = []

    for col_name, label in labels_set.items():
        subject_id = col_name.split("_")[0]
        if (
            col_name in raw_eda.columns
        ):  # Only add labels with corresponding data, EDA dataset lacks one entry which ECG has
            length = raw_eda[col_name].size
            sample_n_series.extend([n for n in range(length)])
            eda_series.extend(raw_eda[col_name])
            ecg_series.extend(raw_ecg[col_name])
            label_values = np.full(length, label)
            if class_type == "b":
                blabel_series.extend(label_values)
            elif class_type == "t":
                tlabel_series.extend(label_values)
            elif class_type == "q":
                qlabel_series.extend(label_values)
            if subject_id not in subject_to_group:
                subject_to_group[subject_id] = group_counter
                group_counter += 1
            subject_series.extend(np.full(length, subject_to_group[subject_id]))
            subjects_in_columns.append(subject_to_group[subject_id])
            task_series.extend(np.full(length, col_name))

data_df = pd.DataFrame({
    # "Subject": subject_series,
    "Task": task_series,
    "Nth-Sample": sample_n_series,
    "EDA": eda_series,
    "ECG": ecg_series})
display(data_df)
only_eda_df = data_df.drop(columns="ECG")
display(only_eda_df)
only_ecg_df = data_df.drop(columns="EDA")
display(only_ecg_df)

tasks_labels = labels["b"].copy()
tasks_labels.drop("r5s8_Counting3", inplace=True)

,Task,Nth-Sample,EDA,ECG
0,2ea4_Breathing,0,32019.942317,10398.583885
1,2ea4_Breathing,1,32019.783775,9831.512610
2,2ea4_Breathing,2,32019.625308,8892.997782
3,2ea4_Breathing,3,32019.466918,7709.957079
4,2ea4_Breathing,4,32019.308609,6422.639589
...,...,...,...,...
20969995,y9z6_Video2,29995,4304.461086,1.998284
20969996,y9z6_Video2,29996,4304.435349,8.936071
20969997,y9z6_Video2,29997,4304.409994,9.970963
20969998,y9z6_Video2,29998,4304.385020,5.986312


,Task,Nth-Sample,EDA
0,2ea4_Breathing,0,32019.942317
1,2ea4_Breathing,1,32019.783775
2,2ea4_Breathing,2,32019.625308
3,2ea4_Breathing,3,32019.466918
4,2ea4_Breathing,4,32019.308609
...,...,...,...
20969995,y9z6_Video2,29995,4304.461086
20969996,y9z6_Video2,29996,4304.435349
20969997,y9z6_Video2,29997,4304.409994
20969998,y9z6_Video2,29998,4304.385020


,Task,Nth-Sample,ECG
0,2ea4_Breathing,0,10398.583885
1,2ea4_Breathing,1,9831.512610
2,2ea4_Breathing,2,8892.997782
3,2ea4_Breathing,3,7709.957079
4,2ea4_Breathing,4,6422.639589
...,...,...,...
20969995,y9z6_Video2,29995,1.998284
20969996,y9z6_Video2,29996,8.936071
20969997,y9z6_Video2,29997,9.970963
20969998,y9z6_Video2,29998,5.986312


In [4]:
# BOTH EDA and ECG
EXTRACT_EDAECG = False
if EXTRACT_EDAECG:
    X = extract_relevant_features(
        data_df,
        y=tasks_labels,
        column_id="Task",
        column_sort="Nth-Sample",
        default_fc_parameters=EfficientFCParameters(),
        n_jobs=6,
    )

In [5]:
if EXTRACT_EDAECG:
    X.to_csv(EDAECG_RELEVANT_FEAT_FILENAME)
else:
    X = pd.read_csv(EDAECG_RELEVANT_FEAT_FILENAME, index_col=0)
display(X)


,"ECG__fft_coefficient__attr_""abs""__coeff_0","ECG__fft_aggregated__aggtype_""skew""","EDA__augmented_dickey_fuller__attr_""usedlag""__autolag_""AIC""","ECG__fft_aggregated__aggtype_""kurtosis""","EDA__agg_linear_trend__attr_""stderr""__chunk_len_50__f_agg_""min""","EDA__agg_linear_trend__attr_""stderr""__chunk_len_10__f_agg_""min""","EDA__agg_linear_trend__attr_""stderr""__chunk_len_5__f_agg_""min""","EDA__agg_linear_trend__attr_""stderr""__chunk_len_10__f_agg_""mean""","EDA__agg_linear_trend__attr_""stderr""__chunk_len_50__f_agg_""mean""","EDA__linear_trend__attr_""stderr""",...,"EDA__fft_coefficient__attr_""angle""__coeff_15","ECG__change_quantiles__f_agg_""mean""__isabs_True__qh_0.8__ql_0.0","EDA__fft_coefficient__attr_""imag""__coeff_23","ECG__fft_coefficient__attr_""abs""__coeff_9","ECG__fft_coefficient__attr_""abs""__coeff_99","EDA__change_quantiles__f_agg_""mean""__isabs_False__qh_0.6__ql_0.4","ECG__cwt_coefficients__coeff_5__w_5__widths_(2, 5, 10, 20)","ECG__agg_linear_trend__attr_""intercept""__chunk_len_50__f_agg_""var""","ECG__change_quantiles__f_agg_""var""__isabs_True__qh_1.0__ql_0.0","EDA__fft_coefficient__attr_""imag""__coeff_9"
2ea4_Breathing,3.856781e+04,3.535058,12.0,17.045377,0.117937,0.010721,0.003798,0.010761,0.120433,0.000340,...,136.038143,44.846211,-2.757051e+05,61981.870812,1.669879e+06,-0.000513,14259.026376,1.009133e+06,28782.415230,1.039277e+06
2ea4_Counting1,9.222276e-10,3.713862,4.0,54.164700,1.444955,0.128578,0.045430,0.128466,1.438134,0.004061,...,-60.942345,41.427420,3.945548e+06,60016.376353,2.023332e+04,0.001014,411.398584,1.185748e+06,28760.649035,2.772885e+06
2ea4_Counting2,3.669811e-09,3.934723,4.0,57.590813,0.967380,0.086028,0.030393,0.085940,0.962058,0.002717,...,-126.854139,36.435008,-1.000245e+06,43262.084795,2.455352e+05,-0.001314,815.105321,1.051532e+06,28653.667095,-1.530502e+06
2ea4_Counting3,2.296474e-09,4.340321,6.0,59.903030,0.445783,0.039885,0.014101,0.039895,0.446500,0.001261,...,-160.554249,43.184946,2.579748e+05,30682.217518,2.344713e+05,-0.230892,1354.446178,1.229961e+06,30013.726321,-1.032751e+07
2ea4_Math,1.413355e-09,5.474369,10.0,69.845463,0.528900,0.047481,0.016794,0.047530,0.531978,0.001503,...,149.337748,36.294536,3.409997e+06,13575.967058,8.622251e+05,-0.003436,-834.234205,1.095058e+06,31644.015222,3.778753e+06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
y9z6_Relax,3.728582e+03,2.275315,13.0,39.993216,0.086170,0.007697,0.002721,0.007697,0.086164,0.000243,...,-93.823552,31.331148,-2.472037e+05,4419.045146,2.687545e+05,-0.012171,13.307987,2.907551e+05,14157.900439,-3.847158e+05
y9z6_Speaking,6.730261e-10,1.567183,4.0,24.260312,0.325079,0.028989,0.010245,0.028978,0.324404,0.000916,...,44.141399,48.419936,6.956725e+05,46473.094448,5.971775e+05,-0.000898,-228.772446,4.485172e+05,18342.977180,1.669520e+06
y9z6_Stroop,2.055458e-09,1.668339,4.0,28.279137,0.206460,0.018324,0.006472,0.018297,0.204817,0.000578,...,124.417732,34.434699,8.879346e+03,54830.956510,4.761965e+05,-0.000446,72.573679,3.154127e+05,13359.750356,9.042556e+05
y9z6_Video1,2.153411e+03,2.881601,36.0,45.335105,0.213687,0.019081,0.006745,0.019080,0.213599,0.000603,...,-72.348865,25.770521,-1.929527e+05,17136.992646,3.262311e+04,-0.062937,-108.850215,2.540104e+05,9830.441636,-9.340153e+05


In [6]:
EXTRACT_EDA = False
if EXTRACT_EDA:
    eda_rel_features = extract_relevant_features(
        only_eda_df,
        y=tasks_labels,
        column_id="Task",
        column_sort="Nth-Sample",
        default_fc_parameters=EfficientFCParameters(),
        n_jobs=6,
    )

In [7]:
if EXTRACT_EDA:
    eda_rel_features.to_csv(EDA_RELEVANT_FEAT_FILENAME)
else:
    eda_rel_features = pd.read_csv(EDA_RELEVANT_FEAT_FILENAME, index_col=0)
display(eda_rel_features)


,"EDA__agg_linear_trend__attr_""stderr""__chunk_len_50__f_agg_""min""","EDA__agg_linear_trend__attr_""stderr""__chunk_len_10__f_agg_""min""","EDA__agg_linear_trend__attr_""stderr""__chunk_len_5__f_agg_""min""","EDA__linear_trend__attr_""stderr""","EDA__agg_linear_trend__attr_""stderr""__chunk_len_5__f_agg_""mean""","EDA__agg_linear_trend__attr_""stderr""__chunk_len_10__f_agg_""mean""","EDA__agg_linear_trend__attr_""stderr""__chunk_len_50__f_agg_""mean""","EDA__agg_linear_trend__attr_""stderr""__chunk_len_10__f_agg_""max""","EDA__agg_linear_trend__attr_""stderr""__chunk_len_5__f_agg_""max""","EDA__agg_linear_trend__attr_""stderr""__chunk_len_50__f_agg_""max""",...,"EDA__agg_linear_trend__attr_""rvalue""__chunk_len_50__f_agg_""mean""","EDA__linear_trend__attr_""rvalue""","EDA__agg_linear_trend__attr_""rvalue""__chunk_len_10__f_agg_""max""","EDA__fft_coefficient__attr_""angle""__coeff_10","EDA__fft_coefficient__attr_""imag""__coeff_22","EDA__agg_linear_trend__attr_""rvalue""__chunk_len_50__f_agg_""max""","EDA__change_quantiles__f_agg_""mean""__isabs_False__qh_0.8__ql_0.4","EDA__change_quantiles__f_agg_""mean""__isabs_False__qh_1.0__ql_0.2","EDA__fft_coefficient__attr_""angle""__coeff_25","EDA__change_quantiles__f_agg_""mean""__isabs_False__qh_0.8__ql_0.2"
2ea4_Breathing,0.117937,0.010721,0.003798,0.000340,0.003804,0.010761,0.120433,0.010802,0.003810,0.122850,...,0.138166,0.138114,0.140014,44.215474,-3.486130e+05,0.148320,0.007812,0.010442,-24.629819,0.005269
2ea4_Counting1,1.444955,0.128578,0.045430,0.004061,0.045412,0.128466,1.438134,0.128354,0.045394,1.431308,...,-0.001207,-0.001205,-0.000897,33.856600,4.188104e+06,0.000478,0.000507,0.256689,103.657951,0.342756
2ea4_Counting2,0.967380,0.086028,0.030393,0.002717,0.030379,0.085940,0.962058,0.085853,0.030366,0.956720,...,-0.810004,-0.809982,-0.810366,126.080262,-2.111307e+05,-0.812086,-0.000677,-0.000234,-88.257570,-0.000312
2ea4_Counting3,0.445783,0.039885,0.014101,0.001261,0.014103,0.039895,0.446500,0.039905,0.014104,0.446998,...,-0.875294,-0.875227,-0.875443,-118.519767,-2.970043e+05,-0.876485,-0.114492,-0.213454,-128.128051,-0.284822
2ea4_Math,0.528900,0.047481,0.016794,0.001503,0.016802,0.047530,0.531978,0.047579,0.016809,0.534843,...,-0.721611,-0.721518,-0.721839,-98.478620,5.226481e+04,-0.723384,-0.002294,-0.000019,56.522580,-0.000837
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
y9z6_Relax,0.086170,0.007697,0.002721,0.000243,0.002721,0.007697,0.086164,0.007696,0.002721,0.086156,...,-0.447281,-0.447275,-0.447405,-77.922950,-2.599745e+05,-0.447992,-0.021035,-0.048195,-95.418127,-0.014052
y9z6_Speaking,0.325079,0.028989,0.010245,0.000916,0.010244,0.028978,0.324404,0.028967,0.010242,0.323709,...,-0.035497,-0.035494,-0.036207,80.228045,8.955983e+05,-0.039384,-0.000395,-0.000127,82.103572,-0.000276
y9z6_Stroop,0.206460,0.018324,0.006472,0.000578,0.006468,0.018297,0.204817,0.018269,0.006463,0.203152,...,-0.868147,-0.868126,-0.868562,-143.395566,2.612847e+05,-0.870518,-0.000094,0.024660,-3.981411,0.032862
y9z6_Video1,0.213687,0.019081,0.006745,0.000603,0.006745,0.019080,0.213599,0.019078,0.006744,0.213508,...,-0.216404,-0.216401,-0.216448,-93.217353,-2.592534e+05,-0.216658,-0.031510,-0.035721,-73.361998,-0.047659


In [8]:
EXTRACT_ECG = False
if EXTRACT_ECG:
    ecg_rel_features = extract_relevant_features(
        only_ecg_df,
        y=tasks_labels,
        column_id="Task",
        column_sort="Nth-Sample",
        default_fc_parameters=EfficientFCParameters(),
        n_jobs=6,
    )

In [9]:
if EXTRACT_ECG:
    ecg_rel_features.to_csv(ECG_RELEVANT_FEAT_FILENAME)
else:
    ecg_rel_features = pd.read_csv(ECG_RELEVANT_FEAT_FILENAME, index_col=0)
display(ecg_rel_features)


,"ECG__fft_coefficient__attr_""abs""__coeff_0","ECG__fft_aggregated__aggtype_""skew""","ECG__fft_aggregated__aggtype_""kurtosis""","ECG__change_quantiles__f_agg_""mean""__isabs_True__qh_0.6__ql_0.4","ECG__change_quantiles__f_agg_""var""__isabs_False__qh_0.6__ql_0.4","ECG__change_quantiles__f_agg_""var""__isabs_False__qh_0.8__ql_0.2","ECG__change_quantiles__f_agg_""var""__isabs_True__qh_0.8__ql_0.2","ECG__change_quantiles__f_agg_""mean""__isabs_True__qh_0.8__ql_0.2","ECG__change_quantiles__f_agg_""mean""__isabs_True__qh_0.6__ql_0.2","ECG__change_quantiles__f_agg_""var""__isabs_True__qh_0.6__ql_0.4",...,"ECG__fft_coefficient__attr_""abs""__coeff_73","ECG__linear_trend__attr_""stderr""",ECG__variance,ECG__standard_deviation,ECG__root_mean_square,ECG__abs_energy,"ECG__agg_linear_trend__attr_""stderr""__chunk_len_5__f_agg_""mean""","ECG__agg_linear_trend__attr_""stderr""__chunk_len_50__f_agg_""max""","ECG__agg_linear_trend__attr_""stderr""__chunk_len_10__f_agg_""mean""","ECG__change_quantiles__f_agg_""var""__isabs_False__qh_1.0__ql_0.8"
2ea4_Breathing,3.856781e+04,3.535058,17.045377,25.792462,1504.356336,5080.609963,3337.904080,41.746862,40.620393,839.550168,...,5.229027e+05,0.000815,1.495114e+06,1222.748650,1222.749325,4.485348e+10,0.008918,0.618079,0.023982,152333.319541
2ea4_Counting1,9.222276e-10,3.713862,54.164700,31.375113,1790.243428,4199.073949,2474.472110,41.528805,43.960663,806.240223,...,3.230400e+06,0.000854,1.640014e+06,1280.630119,1280.630119,4.920041e+10,0.009356,0.647197,0.024924,161687.761977
2ea4_Counting2,3.669811e-09,3.934723,57.590813,27.237470,1326.168593,3489.136895,2175.236066,36.248390,37.820689,584.710140,...,3.696235e+05,0.000838,1.580450e+06,1257.159351,1257.159351,4.741349e+10,0.009184,0.638848,0.024648,157229.706643
2ea4_Counting3,2.296474e-09,4.340321,59.903030,32.734921,1833.358520,4296.643855,2393.780709,43.621820,44.266107,762.268144,...,1.156736e+06,0.000866,1.686854e+06,1298.789515,1298.789515,5.060563e+10,0.009486,0.657760,0.025476,168853.805553
2ea4_Math,1.413355e-09,5.474369,69.845463,26.163356,1281.345027,3461.666104,2193.155822,35.616231,36.250203,597.188630,...,6.292091e+05,0.000872,1.710687e+06,1307.932329,1307.932329,5.132061e+10,0.009552,0.659565,0.025517,172895.923442
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
y9z6_Relax,3.728582e+03,2.275315,39.993216,15.955197,472.849990,902.551862,446.786247,21.374394,21.525032,218.282910,...,1.411136e+05,0.000408,3.744872e+05,611.953587,611.953600,1.123462e+10,0.004379,0.299244,0.011253,53932.869843
y9z6_Speaking,6.730261e-10,1.567183,24.260312,24.835647,1169.412148,3254.926738,2090.902166,34.123537,32.418761,552.629907,...,2.610783e+05,0.000539,6.534074e+05,808.336196,808.336196,1.960222e+10,0.005830,0.325308,0.015300,66096.880368
y9z6_Stroop,2.055458e-09,1.668339,28.279137,21.067075,830.347788,1477.314268,834.390512,25.378243,26.173005,386.568733,...,3.279967e+05,0.000431,4.186327e+05,647.018289,647.018289,1.255898e+10,0.004653,0.304117,0.012290,54416.910607
y9z6_Video1,2.153411e+03,2.881601,45.335105,13.272262,324.117650,801.174301,472.241567,18.141931,17.785333,147.965477,...,1.396112e+05,0.000388,3.380945e+05,581.458941,581.458945,1.014284e+10,0.004198,0.288709,0.010987,44499.989276


In [10]:
# Selecting rows that actually have entries in "labels" file
idx = list(X.merge(labels["b"], left_index=True, right_index=True).index)
y = labels["b"].loc[idx]
x = X.loc[idx]
print(f"Selected {len(x)} entries from X and {len(y)} labels.")


Selected 699 entries from X and 699 labels.


In [ ]:
# Divergence analysis on StressID
import plotly.express as px

perplexity = np.arange(20, 220, 20)
divergence = []
si_Ncomp = 2

for i in perplexity:
    model = TSNE(n_components=si_Ncomp, init="pca", perplexity=i)
    reduced = model.fit_transform(eda_rel_features)
    divergence.append(model.kl_divergence_)

fig = px.line(x=perplexity, y=divergence, markers=True, width=800, height=600)
fig.update_layout(xaxis_title="Perplexity Values", yaxis_title="KL Divergence")
fig.update_traces(line_color="red", line_width=1)
fig.show()

In [49]:
# t-SNE in StressID
# Best N-comp=2, Perp=190 np.arange(10, 200, 10) X
si_Ncomp = 2
si_Perp = 190
BY_CLASS = False
colors = y if BY_CLASS else subjects_in_columns

si_tsne = TSNE(n_components=si_Ncomp, perplexity=si_Perp, random_state=RANDOM_STATE)
si_X_tsne = si_tsne.fit_transform(X)
display(si_tsne.kl_divergence_)

fig_cls = px.scatter(x=si_X_tsne[:, 0], y=si_X_tsne[:, 1], color=y.astype(str), width=800, height=600, opacity=0.5)
fig_cls.update_layout(
    title="StressID dataset t-SNE (TSFresh ECG+EDA features) by Class",
    xaxis_title="1st t-SNE",
    yaxis_title="2nd t-SNE",
)
fig_cls.show()

fig_subj = px.scatter(x=si_X_tsne[:, 0], y=si_X_tsne[:, 1], color=subjects_in_columns, width=800, height=600)
fig_subj.update_layout(
    title="StressID dataset t-SNE (TSFresh ECG+EDA features) by Subject",
    xaxis_title="1st t-SNE",
    yaxis_title="2nd t-SNE",
)
fig_subj.show()

# Picking a small sample of subjects to show
n_subjects = 4
subjects_to_show = np.random.choice(range(len(subject_to_group)), size=n_subjects)
subject_mask = [n in subjects_to_show for n in subjects_in_columns]
filtered_subjects_rows = [str(subject) for subject, flag in zip(subjects_in_columns, subject_mask) if flag]
filtered_classes_rows = [str(label) for label, flag in zip(y.values, subject_mask) if flag]

fig_subj = px.scatter(x=si_X_tsne[:, 0][subject_mask], y=si_X_tsne[:, 1][subject_mask], color=filtered_subjects_rows, symbol=filtered_classes_rows, width=800, height=600)
fig_subj.update_layout(
    title=f"StressID dataset t-SNE (TSFresh ECG+EDA features) by {n_subjects} Subject",
    xaxis_title="1st t-SNE",
    yaxis_title="2nd t-SNE",
)
fig_subj.update_traces(
    marker=dict(size=12, opacity=0.7),
    selector=dict(mode="markers")
)
fig_subj.show()

0.05135215073823929

In [ ]:
# t-SNE in StressID
# Best N-comp=2, Perp=190 np.arange(10, 200, 10) X
si_Ncomp = 2
si_Perp = 190
subjects_to_show = np.random.choice(range(len(subject_to_group)), size=5)
subject_mask = [n in subjects_to_show for n in subjects_in_columns]

si_tsne = TSNE(n_components=si_Ncomp, perplexity=si_Perp, random_state=RANDOM_STATE)

si_X_tsne = si_tsne.fit_transform(X)
display(si_tsne.kl_divergence_)

fig_cls = px.scatter(x=si_X_tsne[:, 0], y=si_X_tsne[:, 1], color=y.astype(str), width=800, height=600, opacity=0.5)
fig_cls.update_layout(
    title="StressID dataset t-SNE (TSFresh ECG+EDA features) by Class",
    xaxis_title="1st t-SNE",
    yaxis_title="2nd t-SNE",
)
fig_cls.show()

In [44]:
# t-SNE in StressID
# Best N-comp=2, Perp=190 np.arange(20, 220, 20) eda_rel_features
si_Ncomp = 2
si_Perp = 190
BY_CLASS = True
colors = y if BY_CLASS else subjects_in_columns

si_tsne = TSNE(n_components=si_Ncomp, perplexity=si_Perp, random_state=RANDOM_STATE)
si_X_tsne = si_tsne.fit_transform(eda_rel_features)
display(si_tsne.kl_divergence_)

fig_cls = px.scatter(x=si_X_tsne[:, 0], y=si_X_tsne[:, 1], color=y.astype(str), width=800, height=600, opacity=0.5)
fig_cls.update_layout(
    title="StressID dataset t-SNE (TSFresh EDA features) by Class",
    xaxis_title="1st t-SNE",
    yaxis_title="2nd t-SNE",
)
fig_cls.show()

fig_subj = px.scatter(x=si_X_tsne[:, 0], y=si_X_tsne[:, 1], color=subjects_in_columns, width=800, height=600)
fig_subj.update_layout(
    title="StressID dataset t-SNE (TSFresh EDA features) by Subject",
    xaxis_title="1st t-SNE",
    yaxis_title="2nd t-SNE",
)
fig_subj.show()

# Picking a small sample of subjects to show
n_subjects = 4
subjects_to_show = np.random.choice(range(len(subject_to_group)), size=n_subjects)
subject_mask = [n in subjects_to_show for n in subjects_in_columns]
filtered_subjects_rows = [str(subject) for subject, flag in zip(subjects_in_columns, subject_mask) if flag]
filtered_classes_rows = [str(label) for label, flag in zip(y.values, subject_mask) if flag]

fig_subj = px.scatter(x=si_X_tsne[:, 0][subject_mask], y=si_X_tsne[:, 1][subject_mask], color=filtered_subjects_rows, symbol=filtered_classes_rows, width=800, height=600)
fig_subj.update_layout(
    title=f"StressID dataset t-SNE (TSFresh EDA features) by {n_subjects} Subject",
    xaxis_title="1st t-SNE",
    yaxis_title="2nd t-SNE",
)
fig_subj.update_traces(
    marker=dict(size=12, opacity=0.7),
    selector=dict(mode="markers")
)
fig_subj.show()


0.05169035494327545

In [43]:
# t-SNE in StressID
# Best N-comp=2, Perp=190 np.arange(15, 210, 15) ecg_rel_features
si_Ncomp = 2
si_Perp = 220
BY_CLASS = False
colors = y if BY_CLASS else subjects_in_columns

si_tsne = TSNE(n_components=si_Ncomp, perplexity=si_Perp, random_state=RANDOM_STATE)
si_X_tsne = si_tsne.fit_transform(ecg_rel_features)
display(si_tsne.kl_divergence_)

fig_cls = px.scatter(x=si_X_tsne[:, 0], y=si_X_tsne[:, 1], color=y, width=800, height=600, opacity=0.5)
fig_cls.update_layout(
    title="StressID dataset t-SNE (TSFresh ECG features) by Class",
    xaxis_title="1st t-SNE",
    yaxis_title="2nd t-SNE",
)
fig_cls.show()

fig_subj = px.scatter(x=si_X_tsne[:, 0], y=si_X_tsne[:, 1], color=subjects_in_columns, width=800, height=600)
fig_subj.update_layout(
    title="StressID dataset t-SNE (TSFresh ECG features) by Subject",
    xaxis_title="1st t-SNE",
    yaxis_title="2nd t-SNE",
)
fig_subj.show()

# Picking a small sample of subjects to show
n_subjects = 4
subjects_to_show = np.random.choice(range(len(subject_to_group)), size=n_subjects)
subject_mask = [n in subjects_to_show for n in subjects_in_columns]
filtered_subjects_rows = [str(subject) for subject, flag in zip(subjects_in_columns, subject_mask) if flag]
filtered_classes_rows = [str(label) for label, flag in zip(y.values, subject_mask) if flag]

fig_subj = px.scatter(x=si_X_tsne[:, 0][subject_mask], y=si_X_tsne[:, 1][subject_mask], color=filtered_subjects_rows, symbol=filtered_classes_rows, width=800, height=600)
fig_subj.update_layout(
    title=f"StressID dataset t-SNE (TSFresh ECG features) by {n_subjects} Subject",
    xaxis_title="1st t-SNE",
    yaxis_title="2nd t-SNE",
)
fig_subj.update_traces(
    marker=dict(size=12, opacity=0.7),
    selector=dict(mode="markers")
)
fig_subj.show()


0.051603127270936966

In [24]:
# t-SNE in StressID
# Best N-comp=3, Perp=90 np.arange(15, 210, 15) X_selected
si_Ncomp = 3
si_Perp = 90
add_noise = False # For when the data points are too close together

si_tsne = TSNE(n_components=si_Ncomp, perplexity=si_Perp, random_state=RANDOM_STATE)
si_X_tsne = si_tsne.fit_transform(X)
display(si_tsne.kl_divergence_)
for_display = si_X_tsne + np.random.normal(0, 0.01, si_X_tsne.shape) if add_noise else si_X_tsne

fig_cls = px.scatter_3d(
    x=for_display[:, 0], y=for_display[:, 1], z=for_display[:, 2], opacity=0.6, color=y, width=800, height=600
)
fig_cls.update_layout(title="StressID dataset t-SNE (TSFresh ECG+EDA features) by Class")
fig_cls.show()
fig_subj = px.scatter_3d(
    x=for_display[:, 0],
    y=for_display[:, 1],
    z=for_display[:, 2],
    color=subjects_in_columns,
    opacity=0.7,
    width=800,
    height=600,
)
fig_subj.update_layout(title="StressID dataset t-SNE (TSFresh ECG+EDA features) by Subject")
fig_subj.show()

0.05273575335741043

In [50]:
# t-SNE in StressID
# Best N-comp=3, Perp=95 np.arange(5, 100, 5) eda_rel_features
si_Ncomp = 3
si_Perp = 95
BY_CLASS = False
colors = y if BY_CLASS else subjects_in_columns

si_tsne = TSNE(n_components=si_Ncomp, perplexity=si_Perp, random_state=RANDOM_STATE)
si_X_tsne = si_tsne.fit_transform(eda_rel_features)
display(si_tsne.kl_divergence_)

fig_cls = px.scatter_3d(
    x=si_X_tsne[:, 0], y=si_X_tsne[:, 1], z=si_X_tsne[:, 2], color=y, opacity=0.5, width=800, height=600
)
fig_cls.update_layout(title="StressID dataset t-SNE (TSFresh EDA features) by Class")
fig_cls.show()
fig_subj = px.scatter_3d(
    x=si_X_tsne[:, 0],
    y=si_X_tsne[:, 1],
    z=si_X_tsne[:, 2],
    color=subjects_in_columns,
    opacity=0.7,
    width=800,
    height=600,
)
fig_subj.update_layout(title="StressID dataset t-SNE (TSFresh EDA features) by Subject")
fig_subj.show()

0.05093998834490776

In [51]:
# t-SNE in StressID
# Best N-comp=3, Perp=95 np.arange(10, 150, 10) ecg_rel_features
si_Ncomp = 3
si_Perp = 100
BY_CLASS = False
colors = y if BY_CLASS else subjects_in_columns

si_tsne = TSNE(n_components=si_Ncomp, perplexity=si_Perp, random_state=RANDOM_STATE)
si_X_tsne = si_tsne.fit_transform(ecg_rel_features)
display(si_tsne.kl_divergence_)

fig_cls = px.scatter_3d(
    x=si_X_tsne[:, 0], y=si_X_tsne[:, 1], z=si_X_tsne[:, 2], color=y, opacity=0.5, width=800, height=600
)
fig_cls.update_layout(title="StressID dataset t-SNE (TSFresh ECG features) by Class")
fig_cls.show()
fig_subj = px.scatter_3d(
    x=si_X_tsne[:, 0],
    y=si_X_tsne[:, 1],
    z=si_X_tsne[:, 2],
    color=subjects_in_columns,
    opacity=0.7,
    width=800,
    height=600,
)
fig_subj.update_layout(title="StressID dataset t-SNE (TSFresh ECG features) by Subject")
fig_subj.show()


0.05462796613574028